# Notebook para testes de coleta otimizada
Construindo estrutura para coleta em larga escala de posts, insights e comentários via Instagram Graph API, suportando fluxos de OAuth via Instagram e via Facebook

Estratégias de otimização da coleta de dados:
1. Posts + Comentários + insights em uma única requisição
2. limit = 100 por página para minimizar número de páginas
3. Cursor-based pagination com salvamento de estado para retomada, em caso de estourar o limite requisições/tempo do app.
4. Monitoramento do Rate Limit via header 
5. Parar a coleta antes do bloqueio e retomar pelo Cursor salvo
6. Batch de múltiplos IDs.

# Dependências e Configuração
dependências:
- python-dotenv
- requests
- tqdm --> mostra barra de progresso

In [1]:
import os
import json
import time
import requests
from datetime import datetime
from tqdm.notebook import tqdm
from dotenv import load_dotenv

In [2]:
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

In [3]:
load_dotenv()

LONG_LIVED_TOKEN = os.getenv("LONG_LIVED_TOKEN")
AUTH_METHOD      = os.getenv("AUTH_METHOD")   # 'facebook' ou 'instagram'
PROFILE_ID       = os.getenv("PROFILE_ID")

In [4]:
BASE_URL = "https://graph.facebook.com/v25.0" if AUTH_METHOD == "facebook" else "https://graph.instagram.com/v25.0"


# Rate limit utils
A cada resposta da API, o header [X-Business-Use-Case](https://developers.facebook.com/docs/graph-api/overview/rate-limiting) informa o percentual do limite horário já consumido.
- Se atingir 85% : aguarda o tempo retornado pelo campo reset_time_duration
- O cursor de paginação é preservado para retomada de onde parou

In [5]:
RATE_LIMIT_THRESHOLD = 85
DEFAULT_BACKOFF = 3600 # espera padrão se o header não informar

In [6]:
def parse_rate_limit(response: requests.Response) -> dict:
    '''
    Extrai informações de rate limit do header X-Business-Use-Case-Usage
    Retorna dict com 'pct_used' e 'reset_in_seconds'
    '''

    header = response.headers.get("X-Business-Use-Case-Usage", "{}")
    try: 
        data = json.loads(header)

        for app_id, usages in data.items():
            if isinstance(usages, list) and usages:
                u = usages[0]
                return {
                "pct_used":         u.get("acc_id_util_pct", 0),
                    "reset_in_seconds": u.get("reset_time_duration", DEFAULT_BACKOFF),
                    "call_type":        u.get("call_type", "unknown"),
                }
    except (json.JSONDecodeError, KeyError):
        pass
    return {"pct_used": 0, "reset_in_seconds": DEFAULT_BACKOFF, "call_type": "unknown"}


In [7]:
def check_and_backoff(response: requests.Response) -> None:
    '''
    Verifica o rate limit e aguarda antes da próxima requisição(se precisar)
    Exibe progresso do backoff em tempo real
    '''
    rate_limit = parse_rate_limit(response)
    pct = rate_limit['pct_used']
    wait = rate_limit['reset_in_seconds']

    if pct >= RATE_LIMIT_THRESHOLD:
        print(f"\nRate limit em {pct}% — aguardando {wait}s para retomar...")
        for remaining in range(wait, 0, -10):
            print(f"{remaining}s restantes...", end="\r")
            time.sleep(min(10, remaining))
        print("\nRetomando coleta.")
    elif pct > 0:
        print(f"Rate limit: {pct:.1f}% usado")



#  Persistência de Cursor - Utils

Cursor after da páginação é salvo em um arquivo .json local após cada página coletada. A fim de garantir que mesmo com o notebook interrompido a coleta pode ser retomada posteriormente

In [8]:
CURSOR_FILE = f"cursor_state_{PROFILE_ID}.json"

In [9]:
def save_cursor(after_cursor: str, page_num: int, collected: int) -> None:
    '''
    Salva o estado atual de paginação em arquivo local.
    '''
    state = {
        "after":     after_cursor,
        "page":      page_num,
        "collected": collected,
        "saved_at":  datetime.utcnow().isoformat(),
    }
    with open(CURSOR_FILE, "w") as f:
        json.dump(state, f, indent=2)

In [10]:
def load_cursor() -> dict | None:
    '''
    Carrega cursor salvo, se existir. Retorna None se não houver estado anterior.
    '''
    if os.path.exists(CURSOR_FILE):
        with open(CURSOR_FILE) as f:
            state = json.load(f)
        print(f"Cursor encontrado: página {state['page']}, "
              f"{state['collected']} posts coletados até {state['saved_at']}")
        return state
    return None


In [11]:
def clear_cursor() -> None:
    '''
    Remove o arquivo de cursor (use ao iniciar nova coleta do zero).
    '''
    if os.path.exists(CURSOR_FILE):
        os.remove(CURSOR_FILE)
        print("Cursor removido. Coleta será iniciada do zero.")

# Requisição base e Field Expansion

Requisição única que traz posts+insights(insights via field expansion do graph api)

- Facebook oauth -> endpoint /ig-user-id/media via graph.facebook.com
- Instagram oauth -> endpoint /me/media via graph/instagram.com

In [12]:
MEDIA_FIELDS = ','.join([
    "id",
    "caption",
    "media_type",
    "media_url",
    "thumbnail_url",
    "permalink",
    "timestamp",
    "like_count",
    "comments_count",
    # Comentários aninhados diretamente na resposta de mídia
    "comments.limit(50){id,text,timestamp,like_count,username,replies.limit(20){id,text,timestamp,username}}",
    # Insights aninhados — disponível apenas para Business/Creator
    "insights.metric(impressions,reach,saved,shares,total_interactions)",
])

In [13]:
def build_media_url(profile_id: str, auth_method: str, after_cursor: str = None) -> str:
    '''
    Monta a URL de coleta de mídia conforme o fluxo de autenticação.
    - facebook: usa o profile_id explícito via graph.facebook.com
    - instagram: usa /me via graph.instagram.com
    '''
    if auth_method == "facebook":
        endpoint = f"{BASE_URL}/{profile_id}/media"
    else:
        endpoint = f"{BASE_URL}/me/media"

    params = {
        "fields":       MEDIA_FIELDS,
        "limit":        100,          # máximo por página para minimizar requisições
        "access_token": LONG_LIVED_TOKEN,
    }
    if after_cursor:
        params["after"] = after_cursor

    req = requests.Request("GET", endpoint, params=params).prepare()
    return req.url

In [14]:
sample_url = build_media_url(PROFILE_ID, AUTH_METHOD)
print("URL de exemplo (sem token):")
print(sample_url.split("access_token")[0] + "access_token=***")

URL de exemplo (sem token):
https://graph.instagram.com/v25.0/me/media?fields=id%2Ccaption%2Cmedia_type%2Cmedia_url%2Cthumbnail_url%2Cpermalink%2Ctimestamp%2Clike_count%2Ccomments_count%2Ccomments.limit%2850%29%7Bid%2Ctext%2Ctimestamp%2Clike_count%2Cusername%2Creplies.limit%2820%29%7Bid%2Ctext%2Ctimestamp%2Cusername%7D%7D%2Cinsights.metric%28impressions%2Creach%2Csaved%2Cshares%2Ctotal_interactions%29&limit=100access_token=***


# Teste de conectividade - Perfil e Primeira Página

In [15]:
def get_profile(profile_id: str, auth_method: str) -> dict:
    '''
    Coleta dados básicos de perfil(profile)
    '''

    if auth_method == 'facebook':
        url = f'{BASE_URL}/{profile_id}'
        fields = 'id,username,name,biography,followers_count,media_count,website'

    else: 
        url = f'{BASE_URL}/me'
        fields = 'id,username,name,biography,followers_count,media_count,website'

    resp = requests.get(url, params={'fields': fields, 'access_token': LONG_LIVED_TOKEN,})

    resp.raise_for_status()
    check_and_backoff(resp)
    return resp.json()

In [17]:
profile = get_profile(PROFILE_ID, AUTH_METHOD)

HTTPError: 400 Client Error: Bad Request for url: https://graph.instagram.com/v25.0/me?fields=id%2Cusername%2Cname%2Cbiography%2Cfollowers_count%2Cmedia_count%2Cwebsite

In [ ]:
print(f"Username       : @{profile.get('username')}")
print(f"Nome           : {profile.get('name')}")
print(f"Seguidores     : {profile.get('followers_count', 'N/A'):,}")
print(f"Total de posts : {profile.get('media_count', 'N/A'):,}")
print(f"Website        : {profile.get('website', 'N/A')}")
print(f"Bio            : {profile.get('biography', '')[:80]}...")

Username       : @technews24.7
Nome           : Notícias Tech 24/7
Seguidores     : 2
Total de posts : 2
Website        : N/A
Bio            : noticias tech...


Primeira página de posts, com field expansion